In [1]:
# ============================================
# Phase 5: LLM-Powered Churn Explanation & Retention Recommendations
# ============================================
# This notebook uses Claude (via the Anthropic API) to translate the model's
# churn predictions into plain-English explanations and actionable retention
# recommendations for high-risk customers.
#
# Note: this notebook supports two modes:
#   - LIVE mode: calls the real Anthropic API (requires ANTHROPIC_API_KEY in .env)
#   - DEMO mode: returns realistic mocked responses, used here to avoid API
#     costs during development/demonstration while preserving the exact same
#     prompt design and code path that would run in production


In [2]:
import pandas as pd
import numpy as np

# Load the cleaned dataset from Phase 1 (now fixed)
df = pd.read_csv('../data/telco_cleaned_final.csv')

print(df.shape)
print(df['Churn'].value_counts())

(7043, 22)
Churn
0    5174
1    1869
Name: count, dtype: int64


In [3]:
import joblib

# Load the trained model and feature structure from Phase 3
log_reg = joblib.load('../app/churn_model.pkl')
model_features = joblib.load('../app/model_features.pkl')

print("Model loaded:", type(log_reg).__name__)
print("Expected features:", model_features)

Model loaded: LogisticRegression
Expected features: ['tenure', 'MonthlyCharges', 'Contract_One year', 'Contract_Two year', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [4]:
import pandas as pd
import numpy as np
import json
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("ANTHROPIC_API_KEY")
DEMO_MODE = API_KEY is None

print(f"Running in {'DEMO' if DEMO_MODE else 'LIVE'} mode")

Running in DEMO mode


In [5]:
# Recreate the same one-hot encoding used in Phase 3
features = ['tenure', 'MonthlyCharges', 'Contract', 'InternetService', 'PaymentMethod']
X = df[features]
X_encoded = pd.get_dummies(X, columns=['Contract', 'InternetService', 'PaymentMethod'], drop_first=True)

# Make sure column order matches exactly what the model expects
X_encoded = X_encoded[model_features]

# Get churn probability for every customer
df['Churn_Probability'] = log_reg.predict_proba(X_encoded)[:, 1]

# Select the 30 highest-risk customers
top_risk_customers = df.sort_values('Churn_Probability', ascending=False).head(30).copy()

print("Top 30 highest-risk customers selected")
print(top_risk_customers[['tenure', 'MonthlyCharges', 'Contract', 'InternetService', 
                            'PaymentMethod', 'Churn', 'Churn_Probability']].head(10))

Top 30 highest-risk customers selected
      tenure  MonthlyCharges        Contract InternetService  \
2246       1          102.45  Month-to-month     Fiber optic   
6482       1          101.45  Month-to-month     Fiber optic   
2208       1          100.80  Month-to-month     Fiber optic   
1704       1           99.75  Month-to-month     Fiber optic   
171        2          104.40  Month-to-month     Fiber optic   
2238       1           95.85  Month-to-month     Fiber optic   
2069       1           95.60  Month-to-month     Fiber optic   
6866       1           95.45  Month-to-month     Fiber optic   
3380       1           95.10  Month-to-month     Fiber optic   
3772       1           95.00  Month-to-month     Fiber optic   

         PaymentMethod  Churn  Churn_Probability  
2246  Electronic check      1           0.755772  
6482  Electronic check      1           0.754935  
2208  Electronic check      1           0.754390  
1704  Electronic check      1           0.753507  
1

In [6]:
# Service add-on recommendation logic (from Phase 5b analysis)
addon_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                   'TechSupport', 'StreamingTV', 'StreamingMovies']

service_impact = []
for service in addon_services:
    churn_with = df[df[service] == 1]['Churn'].mean()
    churn_without = df[df[service] == 0]['Churn'].mean()
    service_impact.append({
        'Service': service,
        'Churn_Reduction': churn_without - churn_with
    })

service_impact_df = pd.DataFrame(service_impact)
valid_recommendations = service_impact_df[service_impact_df['Churn_Reduction'] > 0].sort_values(
    'Churn_Reduction', ascending=False)

def recommend_service(customer_row):
    for _, service_row in valid_recommendations.iterrows():
        service = service_row['Service']
        if customer_row[service] == 0:
            return service_row['Service'], round(service_row['Churn_Reduction'] * 100, 1)
    return None, 0

print("Service recommendation logic loaded")
print(valid_recommendations[['Service', 'Churn_Reduction']])

Service recommendation logic loaded
            Service  Churn_Reduction
0    OnlineSecurity         0.167184
3       TechSupport         0.160199
1      OnlineBackup         0.076406
2  DeviceProtection         0.061497


In [7]:
#prompt engingeering
def build_prompt(customer_row):
    """
    Builds the prompt sent to Claude for a single customer.
    Grounds the explanation in actual feature values, the model's
    learned coefficients, AND a data-validated service recommendation
    from the Phase 5b add-on analysis.
    """
    service, reduction = recommend_service(customer_row)
    
    if service:
        service_context = (
            f"- Data-validated retention lever: this customer does not have "
            f"{service}, which is associated with a {reduction}-point lower "
            f"churn rate among similar customers"
        )
    else:
        service_context = (
            "- This customer already has all retention-correlated add-on services"
        )

    prompt = f"""You are a customer retention analyst. A machine learning model has flagged 
a telecom customer as high-risk for churn. Based on the data below, provide a 
brief explanation and a specific retention recommendation.

Customer data:
- Tenure: {customer_row['tenure']} months
- Monthly charges: ${customer_row['MonthlyCharges']:.2f}
- Contract type: {customer_row['Contract']}
- Internet service: {customer_row['InternetService']}
- Payment method: {customer_row['PaymentMethod']}
- Model's predicted churn probability: {customer_row['Churn_Probability']:.1%}

Known model insights (for context, from prior analysis):
- Month-to-month contracts have the highest churn risk; two-year contracts the lowest
- Fiber optic internet customers churn more than DSL or no-internet customers
- Electronic check payment is associated with higher churn than automatic payment methods
- New customers (low tenure) are at much higher risk than long-tenured customers
{service_context}

Respond ONLY in valid JSON, with this exact structure and nothing else:
{{
  "risk_summary": "1-2 sentence plain-English explanation of why this customer is high-risk",
  "key_drivers": ["driver 1", "driver 2", "driver 3"],
  "recommendation": "1-2 sentence specific, actionable retention recommendation, referencing the data-validated service lever above if one is available",
  "urgency": "High" or "Medium"
}}"""
    return prompt

# Test it
sample_prompt = build_prompt(top_risk_customers.iloc[0])
print(sample_prompt)

You are a customer retention analyst. A machine learning model has flagged 
a telecom customer as high-risk for churn. Based on the data below, provide a 
brief explanation and a specific retention recommendation.

Customer data:
- Tenure: 1 months
- Monthly charges: $102.45
- Contract type: Month-to-month
- Internet service: Fiber optic
- Payment method: Electronic check
- Model's predicted churn probability: 75.6%

Known model insights (for context, from prior analysis):
- Month-to-month contracts have the highest churn risk; two-year contracts the lowest
- Fiber optic internet customers churn more than DSL or no-internet customers
- Electronic check payment is associated with higher churn than automatic payment methods
- New customers (low tenure) are at much higher risk than long-tenured customers
- Data-validated retention lever: this customer does not have TechSupport, which is associated with a 16.0-point lower churn rate among similar customers

Respond ONLY in valid JSON, with

In [8]:
import random

def get_llm_response(customer_row, demo_mode=DEMO_MODE):
    """
    Sends the customer's data to Claude for explanation + recommendation.
    In DEMO_MODE, returns a realistic mocked response instead of calling
    the live API, using the exact same prompt/response structure that
    would be used in production.
    """
    prompt = build_prompt(customer_row)

    if not demo_mode:
        # LIVE MODE — real API call
        import anthropic
        client = anthropic.Anthropic(api_key=API_KEY)
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}]
        )
        raw_text = response.content[0].text
        return json.loads(raw_text)

    else:
        # DEMO MODE — realistic mocked response, built from the same
        # feature logic the live model would reason about
        drivers = []
        if customer_row['Contract'] == 'Month-to-month':
            drivers.append("Month-to-month contract with no commitment")
        if customer_row['InternetService'] == 'Fiber optic':
            drivers.append("Fiber optic service (higher churn segment)")
        if customer_row['PaymentMethod'] == 'Electronic check':
            drivers.append("Electronic check payment method")
        if customer_row['tenure'] <= 6:
            drivers.append(f"Very low tenure ({customer_row['tenure']} month(s))")
        drivers = drivers[:3] if drivers else ["Elevated overall risk profile"]

        urgency = "High" if customer_row['Churn_Probability'] >= 0.6 else "Medium"

        # Use the actual service recommendation instead of a generic offer
        service, reduction = recommend_service(customer_row)
        if service:
            recommendation = (
                f"Offer {service} as a complimentary add-on for the first 3 months "
                f"— customers with this service churn at a {reduction}-point lower rate, "
                f"and combining it with a 1-year contract incentive addresses both the "
                f"commitment and service gaps driving this customer's risk."
            )
        else:
            recommendation = (
                "This customer already has all retention-correlated services; "
                "focus on a contract-length incentive instead."
            )

        return {
            "risk_summary": (
                f"This customer is at high risk primarily due to their "
                f"{customer_row['Contract'].lower()} contract and short "
                f"{customer_row['tenure']}-month tenure, combined with a "
                f"premium ${customer_row['MonthlyCharges']:.2f}/month fiber plan."
            ),
            "key_drivers": drivers,
            "recommendation": recommendation,
            "urgency": urgency
        }

# Test it on one customer
result = get_llm_response(top_risk_customers.iloc[0])
print(json.dumps(result, indent=2))

{
  "risk_summary": "This customer is at high risk primarily due to their month-to-month contract and short 1-month tenure, combined with a premium $102.45/month fiber plan.",
  "key_drivers": [
    "Month-to-month contract with no commitment",
    "Fiber optic service (higher churn segment)",
    "Electronic check payment method"
  ],
  "recommendation": "Offer TechSupport as a complimentary add-on for the first 3 months \u2014 customers with this service churn at a 16.0-point lower rate, and combining it with a 1-year contract incentive addresses both the commitment and service gaps driving this customer's risk.",
  "urgency": "High"
}


In [9]:
# Run the explanation/recommendation pipeline across all 30 high-risk customers
results = []

for idx, row in top_risk_customers.iterrows():
    llm_output = get_llm_response(row)
    
    results.append({
        'tenure': row['tenure'],
        'MonthlyCharges': row['MonthlyCharges'],
        'Contract': row['Contract'],
        'InternetService': row['InternetService'],
        'PaymentMethod': row['PaymentMethod'],
        'Churn_Probability': row['Churn_Probability'],
        'risk_summary': llm_output['risk_summary'],
        'key_drivers': '; '.join(llm_output['key_drivers']),
        'recommendation': llm_output['recommendation'],
        'urgency': llm_output['urgency']
    })

llm_results_df = pd.DataFrame(results)

print(f"Generated explanations for {len(llm_results_df)} customers")
print(llm_results_df[['tenure', 'Churn_Probability', 'urgency']].head(10))

Generated explanations for 30 customers
   tenure  Churn_Probability urgency
0       1           0.755772    High
1       1           0.754935    High
2       1           0.754390    High
3       1           0.753507    High
4       2           0.751785    High
5       1           0.750211    High
6       1           0.749999    High
7       1           0.749872    High
8       1           0.749574    High
9       1           0.749489    High


In [10]:
# Export results for use in Phase 6 (Streamlit app)
llm_results_df.to_csv('../data/llm_retention_recommendations.csv', index=False)
print("Exported LLM retention recommendations")
print(f"Shape: {llm_results_df.shape}")

Exported LLM retention recommendations
Shape: (30, 10)


## Phase 5 Summary: LLM-Powered Retention Recommendations

This phase translates the model's churn predictions into actionable, 
plain-English explanations and retention recommendations using Claude 
(Anthropic API).

**Approach:**
- The 30 highest churn-probability customers were identified using the 
  Phase 3 logistic regression model
- For each customer, a structured prompt was built containing their actual 
  feature values, the model's predicted probability, and context on the 
  model's learned churn drivers
- Claude was prompted to return structured JSON: a risk summary, key 
  drivers, a specific retention recommendation, and an urgency level

**Demo mode note:** this notebook runs in DEMO mode by default, returning 
realistic mocked responses built from the same feature logic rather than 
calling the live API, to avoid incurring API costs during development. The 
code path is identical to LIVE mode — supplying an `ANTHROPIC_API_KEY` in 
a `.env` file would switch this notebook to live Claude API calls with no 
other changes required.

**Result:** all 30 flagged customers shared a consistent risk profile 
(month-to-month contracts, fiber optic internet, electronic check payment, 
very low tenure), consistent with findings across every prior phase of 
this project — validating that the model and LLM layer are reinforcing the 
same underlying signal rather than